# SIBA-Jittor 现场演示

本 Notebook 只组织真实代码、真实日志和真实输出，不生成替代数据。完整 60 轮训练与 706 对测试结果已经保存在仓库中；现场训练只运行少量真实训练步，用于展示代码可执行性，不作为论文指标。


## 1. 环境与源码完整性


In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path('/root/autodl-tmp/SIBA-Jittor')
print((PROJECT_ROOT / 'logs/environment/reproduction_environment.txt').read_text())
source_audit = json.loads((PROJECT_ROOT / 'docs/source_audit_final_20260728.json').read_text())
print(json.dumps(source_audit['comparison'], ensure_ascii=False, indent=2))


## 2. 数值对齐的分层结论


In [ ]:
assessment = json.loads((PROJECT_ROOT / 'logs/alignment/alignment_assessment_final_20260728.json').read_text())
assessment['measured'], assessment['conclusions']


## 3. 真实训练图像上的短训练演示


In [ ]:
!cd "{PROJECT_ROOT}" && bash scripts/start_demo_training_screen.sh
print('在终端执行 screen -r kk，可连续查看每一步真实损失。')


In [ ]:
shared_initial = PROJECT_ROOT / 'logs/demo_shared_initial/SIBA_seed2025_initial.json'
print(shared_initial.read_text())


In [ ]:
latest_demo = Path((PROJECT_ROOT / 'logs/latest_demo.txt').read_text().strip())
print((latest_demo / 'train.log').read_text())


## 4. 完整 60 轮训练日志与损失曲线


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(PROJECT_ROOT / 'results/training_analysis_20260727_siba_official_protocol/loss_curve.png')))
print((PROJECT_ROOT / 'results/training_analysis_20260727_siba_official_protocol/training_log_summary.json').read_text())


## 5. Jittor 推理


In [ ]:
JITTOR_CHECKPOINT = PROJECT_ROOT / 'checkpoints/jittor_msrs_roadscene_60e_20260727_siba_official_protocol/07-27-04-52/SIBA_epoch60.pkl'
DEMO_OUTPUT = PROJECT_ROOT / 'results/demo_jittor_tno'
JITTOR_PYTHON = Path('/root/autodl-tmp/envs/JittorDome/bin/python')
!"{JITTOR_PYTHON}" "{PROJECT_ROOT / 'tools/run_inference.py'}" --framework jittor --checkpoint "{JITTOR_CHECKPOINT}" --data-dir /root/autodl-tmp/datasets/SIBA/test/TNO --output "{DEMO_OUTPUT}" --use-cuda --warmup-runs 3 --timing-mode synchronized


## 6. 真实融合结果与指标


In [ ]:
from PIL import Image as PILImage
from IPython.display import display

images = sorted(DEMO_OUTPUT.glob('*.png')) + sorted(DEMO_OUTPUT.glob('*.jpg'))
print('新生成融合图数量:', len(images))
for path in images[:3]:
    print(path.name)
    display(PILImage.open(path))


In [ ]:
import pandas as pd

metrics = pd.read_csv(PROJECT_ROOT / 'results/metrics_20260727_siba_official_protocol/metrics_summary.csv')
metrics[metrics['dataset'] == 'TNO']


## 7. PyTorch 与 Jittor 输出一致性


In [ ]:
for dataset in ['MSRS', 'M3FD_2x', 'TNO']:
    report = json.loads((PROJECT_ROOT / f'results/output_alignment_20260727_siba_official_protocol/{dataset}/summary.json').read_text())
    print(dataset, report)
